In [1]:
from google.colab import files

uploaded = files.upload()

Saving baseline_features.tsv to baseline_features.tsv


In [2]:
import pandas as pd

df = pd.read_csv("baseline_features.tsv", sep="\t")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isna().sum())

Shape: (945996, 20)

Columns:
['s1_entity_id', 'candidate_entity_id', 'candidate_source', 'label', 'name_exact', 'name_jaccard', 'name_token_overlap', 'name_levenshtein_ratio', 'name_length_difference', 'name_token_count_diff', 'address_exact', 'address_jaccard', 'address_token_overlap', 'address_levenshtein_ratio', 'address_length_difference', 'address_token_count_diff', 'address_missing', 'country_match', 'source_is_s2', 'source_is_s3']

First 5 rows:


,s1_entity_id,candidate_entity_id,candidate_source,label,name_exact,name_jaccard,name_token_overlap,name_levenshtein_ratio,name_length_difference,name_token_count_diff,address_exact,address_jaccard,address_token_overlap,address_levenshtein_ratio,address_length_difference,address_token_count_diff,address_missing,country_match,source_is_s2,source_is_s3
0,S1-965667,S3-860443364,S3,1,0,0.60,0.75,0.571429,0.107143,0,0,0.000000,0.000000,0.000000,0.000000,0,1,1,0,1
1,S1-965667,S3-11291185,S3,1,0,0.00,0.00,0.821429,0.071429,2,0,0.222222,0.333333,0.450000,0.250000,1,0,1,0,1
2,S1-965667,S2-681193310,S2,1,0,0.60,0.75,0.928571,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,0,1,1,1,0
3,S1-965667,S3-775321672,S3,1,0,0.00,0.00,0.107143,0.750000,3,0,0.333333,0.428571,0.608696,0.347826,2,0,1,0,1
4,S1-965667,S2-743505751,S2,1,0,0.75,0.75,0.857143,0.142857,1,0,0.000000,0.000000,0.000000,0.000000,0,1,1,1,0



Label distribution:
label
0    599999
1    345997
Name: count, dtype: int64

Missing values:
s1_entity_id                 0
candidate_entity_id          0
candidate_source             0
label                        0
name_exact                   0
name_jaccard                 0
name_token_overlap           0
name_levenshtein_ratio       0
name_length_difference       0
name_token_count_diff        0
address_exact                0
address_jaccard              0
address_token_overlap        0
address_levenshtein_ratio    0
address_length_difference    0
address_token_count_diff     0
address_missing              0
country_match                0
source_is_s2                 0
source_is_s3                 0
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

# Get unique S1 entities
unique_s1 = df["s1_entity_id"].unique()

print("Total unique S1 entities:", len(unique_s1))

# 80% S1 entities for training, 20% for validation
train_s1, val_s1 = train_test_split(
    unique_s1,
    test_size=0.20,
    random_state=42
)

print("Training S1 entities:", len(train_s1))
print("Validation S1 entities:", len(val_s1))

# Create the actual row-level datasets
train_df = df[df["s1_entity_id"].isin(train_s1)].copy()
val_df = df[df["s1_entity_id"].isin(val_s1)].copy()

print("\nTraining pair rows:", len(train_df))
print("Validation pair rows:", len(val_df))

# Safety check: no S1 overlap
overlap = set(train_s1) & set(val_s1)

print("\nS1 overlap:", len(overlap))

Total unique S1 entities: 100000
Training S1 entities: 80000
Validation S1 entities: 20000

Training pair rows: 756638
Validation pair rows: 189358

S1 overlap: 0


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The 16 ML feature columns
feature_cols = [
    "name_exact",
    "name_jaccard",
    "name_token_overlap",
    "name_levenshtein_ratio",
    "name_length_difference",
    "name_token_count_diff",
    "address_exact",
    "address_jaccard",
    "address_token_overlap",
    "address_levenshtein_ratio",
    "address_length_difference",
    "address_token_count_diff",
    "address_missing",
    "country_match",
    "source_is_s2",
    "source_is_s3"
]

X_train = train_df[feature_cols]
y_train = train_df["label"]

X_val = val_df[feature_cols]
y_val = val_df["label"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

# Logistic Regression pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

print("\nTraining Logistic Regression...")
model.fit(X_train, y_train)

print("✅ Model training completed.")

X_train: (756638, 16)
y_train: (756638,)
X_val: (189358, 16)
y_val: (189358,)

Training Logistic Regression...
✅ Model training completed.


In [5]:
# Predict probability of being a true match
val_df = val_df.copy()

val_df["match_probability"] = model.predict_proba(
    X_val
)[:, 1]

print("Probability prediction completed.")

print("\nFirst 10 predictions:")
display(
    val_df[
        [
            "s1_entity_id",
            "candidate_entity_id",
            "candidate_source",
            "label",
            "match_probability"
        ]
    ].head(10)
)

print("\nProbability statistics:")
print(val_df["match_probability"].describe())

Probability prediction completed.

First 10 predictions:


,s1_entity_id,candidate_entity_id,candidate_source,label,match_probability
29,S1-29845983,S3-588502663,S3,1,1.0
30,S1-29845983,S2-648035184,S2,1,1.0
86,S1-264156494,S2-184087846,S2,1,1.0
87,S1-264156494,S3-544213330,S3,1,1.0
88,S1-264156494,S3-562014765,S3,1,1.0
143,S1-72444401,S2-9053863,S2,1,1.0
144,S1-72444401,S2-503949813,S2,1,1.0
145,S1-72444401,S3-505153321,S3,1,1.0
146,S1-72444401,S2-472011237,S2,1,1.0
147,S1-72444401,S2-973800896,S2,1,1.0



Probability statistics:
count    1.893580e+05
mean     3.661346e-01
std      4.794257e-01
min      1.003903e-08
25%      2.807329e-06
50%      1.067584e-04
75%      1.000000e+00
max      1.000000e+00
Name: match_probability, dtype: float64


In [6]:
import numpy as np
import pandas as pd

def f05_score(precision, recall):
    """
    F0.5 score: precision is weighted more than recall.
    """
    if precision == 0 and recall == 0:
        return 0.0

    return (1.25 * precision * recall) / (0.25 * precision + recall)


def evaluate_entity_level(df, threshold):
    """
    Calculate macro Precision, Recall and F0.5
    at the Source-1 entity level.
    """

    precisions = []
    recalls = []
    f05_scores = []

    for s1_id, group in df.groupby("s1_entity_id"):

        # Ground-truth matched candidates
        truth = set(
            group.loc[group["label"] == 1, "candidate_entity_id"]
        )

        # Predicted matched candidates
        predicted = set(
            group.loc[
                group["match_probability"] >= threshold,
                "candidate_entity_id"
            ]
        )

        # Special case: true zero-match entity
        if len(truth) == 0:
            if len(predicted) == 0:
                precision = 1.0
                recall = 1.0
                f05 = 1.0
            else:
                precision = 0.0
                recall = 0.0
                f05 = 0.0

        else:
            true_positive = len(truth & predicted)

            precision = (
                true_positive / len(predicted)
                if len(predicted) > 0
                else 0.0
            )

            recall = true_positive / len(truth)

            f05 = f05_score(precision, recall)

        precisions.append(precision)
        recalls.append(recall)
        f05_scores.append(f05)

    return {
        "precision": np.mean(precisions),
        "recall": np.mean(recalls),
        "f0.5": np.mean(f05_scores),
    }


# Test thresholds
thresholds = np.arange(0.50, 0.96, 0.05)

results = []

for threshold in thresholds:

    metrics = evaluate_entity_level(
        val_df,
        threshold
    )

    results.append({
        "threshold": round(float(threshold), 2),
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f0.5": metrics["f0.5"]
    })


threshold_results = pd.DataFrame(results)

print("Entity-level threshold results:")
display(threshold_results)

Entity-level threshold results:


,threshold,precision,recall,f0.5
0,0.50,0.996981,0.994677,0.996095
1,0.55,0.997249,0.994190,0.996188
2,0.60,0.997681,0.993826,0.996436
3,0.65,0.997883,0.993163,0.996419
4,0.70,0.998088,0.992588,0.996430
5,0.75,0.998323,0.992028,0.996463
6,0.80,0.998503,0.991237,0.996380
7,0.85,0.998586,0.990081,0.996116
8,0.90,0.998586,0.988453,0.995684
9,0.95,0.998331,0.984983,0.994544


In [7]:
best_row = threshold_results.loc[
    threshold_results["f0.5"].idxmax()
]

print("Best threshold based on validation F0.5:")
print(best_row)

Best threshold based on validation F0.5:
threshold    0.750000
precision    0.998323
recall       0.992028
f0.5         0.996463
Name: 5, dtype: float64


In [8]:
best_threshold = 0.75

val_df["predicted_label"] = (
    val_df["match_probability"] >= best_threshold
).astype(int)

false_positives = val_df[
    (val_df["label"] == 0) &
    (val_df["predicted_label"] == 1)
]

false_negatives = val_df[
    (val_df["label"] == 1) &
    (val_df["predicted_label"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

False Positives: 78
False Negatives: 548


In [9]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(
    val_df["label"],
    val_df["predicted_label"]
))

[[119922     78]
 [   548  68810]]


In [10]:
print("FALSE POSITIVES:", len(false_positives))
print("FALSE NEGATIVES:", len(false_negatives))

print("\nTop False Positives:")
display(
    false_positives[
        [
            "s1_entity_id",
            "candidate_entity_id",
            "candidate_source",
            "match_probability",
            "name_jaccard",
            "name_token_overlap",
            "name_levenshtein_ratio",
            "address_jaccard",
            "address_token_overlap",
            "address_levenshtein_ratio",
            "country_match"
        ]
    ].sort_values(
        "match_probability",
        ascending=False
    ).head(20)
)

print("\nTop False Negatives:")
display(
    false_negatives[
        [
            "s1_entity_id",
            "candidate_entity_id",
            "candidate_source",
            "match_probability",
            "name_jaccard",
            "name_token_overlap",
            "name_levenshtein_ratio",
            "address_jaccard",
            "address_token_overlap",
            "address_levenshtein_ratio",
            "country_match"
        ]
    ].sort_values(
        "match_probability",
        ascending=True
    ).head(20)
)

FALSE POSITIVES: 78
FALSE NEGATIVES: 548

Top False Positives:


,s1_entity_id,candidate_entity_id,candidate_source,match_probability,name_jaccard,name_token_overlap,name_levenshtein_ratio,address_jaccard,address_token_overlap,address_levenshtein_ratio,country_match
600007,S1-953857321,S2-954553844,S2,0.999769,0.000000,0.000000,0.240000,0.428571,0.600000,0.709677,1
622770,S1-174437501,S2-510227398,S2,0.999648,0.600000,0.600000,0.677419,0.153846,0.222222,0.255814,1
417618,S1-73305823,S2-463609765,S2,0.999462,0.000000,0.000000,0.250000,0.444444,0.571429,0.631579,1
506936,S1-715246203,S2-88426545,S2,0.998924,0.400000,0.500000,0.606061,0.266667,0.400000,0.500000,1
742047,S1-608464249,S3-575072541,S3,0.998665,0.181818,0.222222,0.522727,0.285714,0.363636,0.420290,1
698311,S1-533696473,S2-80454215,S2,0.998237,0.000000,0.000000,0.208333,0.428571,0.600000,0.685714,1
469495,S1-245815659,S2-76283224,S2,0.997642,0.000000,0.000000,0.074074,0.375000,0.428571,0.609756,1
353126,S1-206286193,S2-822877932,S2,0.997629,0.285714,0.400000,0.552632,0.272727,0.300000,0.459016,1
934164,S1-617315720,S2-176748489,S2,0.997447,0.000000,0.000000,0.088889,0.266667,0.363636,0.236559,1
670428,S1-246031930,S2-225687777,S2,0.996877,0.000000,0.000000,0.181818,0.400000,0.571429,0.651163,1



Top False Negatives:


,s1_entity_id,candidate_entity_id,candidate_source,match_probability,name_jaccard,name_token_overlap,name_levenshtein_ratio,address_jaccard,address_token_overlap,address_levenshtein_ratio,country_match
154435,S1-750471565,S3-703243215,S3,0.000054,0.000000,0.000000,0.083333,0.076923,0.142857,0.204545,1
157295,S1-174129319,S3-671553961,S3,0.000237,0.000000,0.000000,0.185185,0.111111,0.200000,0.233333,1
216690,S1-29030470,S3-807320138,S3,0.000528,0.000000,0.000000,0.250000,0.111111,0.200000,0.275000,1
312662,S1-997552072,S3-375406499,S3,0.000582,0.500000,0.666667,0.400000,0.000000,0.000000,0.500000,1
110898,S1-712038877,S2-554777628,S2,0.000845,0.000000,0.000000,0.076923,0.111111,0.181818,0.262295,1
155951,S1-467406102,S3-839602817,S3,0.001168,0.000000,0.000000,0.210526,0.111111,0.200000,0.517241,1
226755,S1-571241238,S2-66190723,S2,0.001448,0.000000,0.000000,0.066667,0.200000,0.285714,0.351351,1
231997,S1-660808866,S3-316169613,S3,0.001775,0.000000,0.000000,0.000000,0.083333,0.117647,0.178218,1
242667,S1-967154708,S3-489509960,S3,0.002581,0.000000,0.000000,0.230769,0.100000,0.166667,0.400000,1
242755,S1-893366889,S3-767494828,S3,0.002763,0.333333,0.500000,0.363636,0.090909,0.166667,0.411765,1


In [11]:
# Inspect Logistic Regression coefficients

logreg = model.named_steps["logreg"]

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logreg.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

display(
    coef_df
    .sort_values("abs_coefficient", ascending=False)
)

,feature,coefficient,abs_coefficient
1,name_jaccard,15.457769,15.457769
2,name_token_overlap,-10.234921,10.234921
7,address_jaccard,8.295126,8.295126
8,address_token_overlap,4.990586,4.990586
3,name_levenshtein_ratio,2.583760,2.583760
12,address_missing,2.094726,2.094726
10,address_length_difference,1.625861,1.625861
9,address_levenshtein_ratio,1.259454,1.259454
5,name_token_count_diff,0.921975,0.921975
6,address_exact,-0.524792,0.524792


In [12]:
# Compare average feature values for positive and negative pairs

comparison = train_df.groupby("label")[feature_cols].mean().T

comparison.columns = ["negative_mean", "positive_mean"]

comparison["difference"] = (
    comparison["positive_mean"] -
    comparison["negative_mean"]
)

display(
    comparison.sort_values(
        "difference",
        ascending=False
    )
)


,negative_mean,positive_mean,difference
address_token_overlap,0.024770,0.673478,0.648708
name_token_overlap,0.045421,0.660522,0.615101
name_jaccard,0.029921,0.614565,0.584644
address_jaccard,0.015171,0.596879,0.581708
name_levenshtein_ratio,0.207465,0.717453,0.509988
address_levenshtein_ratio,0.204232,0.638780,0.434547
name_exact,0.000000,0.216394,0.216394
address_exact,0.000000,0.082570,0.082570
source_is_s3,0.500001,0.517277,0.017276
address_missing,0.033192,0.044151,0.010960


In [13]:
reduced_features = [
    "name_exact",
    "name_jaccard",
    "name_token_overlap",
    "name_levenshtein_ratio",
    "name_length_difference",
    "name_token_count_diff",
    "address_exact",
    "address_jaccard",
    "address_token_overlap",
    "address_levenshtein_ratio",
    "address_length_difference",
    "address_token_count_diff",
    "address_missing",
    "source_is_s3"
]

X_train_v2 = train_df[reduced_features]
X_val_v2 = val_df[reduced_features]

model_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

print("Training Baseline V2...")

model_v2.fit(
    X_train_v2,
    train_df["label"]
)

val_df["probability_v2"] = model_v2.predict_proba(
    X_val_v2
)[:, 1]

print("✅ Baseline V2 trained.")

Training Baseline V2...
✅ Baseline V2 trained.


In [14]:
results_v2 = []

for threshold in np.arange(0.50, 0.96, 0.05):

    temp_df = val_df.copy()

    temp_df["match_probability"] = temp_df["probability_v2"]

    metrics = evaluate_entity_level(
        temp_df,
        round(float(threshold), 2)
    )

    results_v2.append({
        "threshold": round(float(threshold), 2),
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f0.5": metrics["f0.5"]
    })

results_v2 = pd.DataFrame(results_v2)

display(results_v2)

,threshold,precision,recall,f0.5
0,0.50,0.996880,0.994776,0.996042
1,0.55,0.997225,0.994250,0.996180
2,0.60,0.997710,0.993795,0.996456
3,0.65,0.997881,0.993088,0.996404
4,0.70,0.998047,0.992580,0.996393
5,0.75,0.998242,0.991925,0.996379
6,0.80,0.998489,0.991032,0.996309
7,0.85,0.998536,0.989834,0.996013
8,0.90,0.998561,0.988177,0.995590
9,0.95,0.998291,0.984599,0.994410


In [15]:
best_v2 = results_v2.loc[
    results_v2["f0.5"].idxmax()
]

print("Baseline V2 best result:")
print(best_v2)

Baseline V2 best result:
threshold    0.600000
precision    0.997710
recall       0.993795
f0.5         0.996456
Name: 2, dtype: float64
